In [ ]:
import os
from google.oauth2 import service_account
from googleapiclient.discovery import build
from openai import OpenAI

# Constants
SPREADSHEET_ID = '1lmVQ8jnKWYowsfEG89FiiLIEAIkBs7uhshuOvMQ1tZc'  
SHEET_NAME = 'Sheet2'
CREDENTIALS_FILE = 'url-to-email-445616-cebe4868914f.json'  
OPENAI_API_KEY = ""
# Category ranking
category_ranking = {
	"TESTIMONIALS": 1,
	"COURSES": 2,
	"SERVICES": 3,
	"WEBINAR": 4,
	"PODCAST": 5,
	"EBOOK": 6,
	"RECENT_BLOG": 7,
	"ABOUT_US": 8,
	"SHOP": 9
}

# Column indices for categories (0-based, A=0, B=1, ..., M=12, N=13, ..., U=20)
category_indices = {
	"ABOUT_US": 12,  # M
	"EBOOK": 13,     # N
	"COURSES": 14,   # O
	"RECENT_BLOG": 15, # P
	"TESTIMONIALS": 16, # Q
	"WEBINAR": 17,   # R
	"SERVICES": 18,  # S
	"PODCAST": 19,   # T
	"SHOP": 20       # U (excluded)
}

# Output column letters
output_columns = {
	"Email 1": "W",
	"Email 1 Data Point": "X",
	"Subsequence 1": "Y",
	"Subsequence 1 Data Point": "Z",
	"Subsequence 2": "AA",
	"Subsequence 2 Data Point": "AB",
	"Subsequence 3": "AC",
	"Subsequence 3 Data Point": "AD",
	"Subsequence 4": "AE",
	"Subsequence 4 Data Point": "AF",
	"Email 2": "AG",
	"Email 3": "AH"
}

# Prompt templates
prompt_templates = {
"Email 1": {
	"TESTIMONIALS": """
		[FIRST NAME] - Saw your work with [PERSONALISATION - SOME INSIGHT FROM ANY OF THE TESTIMONIALS OR REVIEWS] — love those results!

		Looks like you can help a ton more folks make the same leap.

		Assuming, if we could 10X your exposure in 45 days, without you lifting a finger…

		We’ll do this by using the insights from your existing stuff and feature them on our network of High Traffic Pages.

		It can build authority and turn strangers into buyers fast, for your paid programs.

		One client saw a 287% Increase in online course purchases. 

		Would you like to know how?
	""",
	"COURSES": """
    [First Name] - Just looked in your [PERSONALISATION - Name of the course or the program, suffix with the word course or program as per the data point and if it feels necessary].  

		Sooo many people could use that!

		Assuming, If we could take the insights & frameworks from it and feature them on our network of High Traffic Pages.

		One client saw a 287% Increase in online course purchases. 

		There’s so much value here, people would go from strangers to flocking to ______  real fast.

		Would you like to know more?
	""",
  "SERVICES": """
    [Fist Name] - Just checked out your [PERSONALISATION - Some insight from their data point] page . 

		Sooo many [PERSONALISATION - Think what could be the ICP for this service and add that ICP here in just keywords] could use that!

		Assuming, If we could take the insights & frameworks from it and feature them on our network of High Traffic Pages.

		One client saw a 287% Increase in online course purchases. 

		There’s so much value here, people would go from strangers to flocking to [PERSONALISATION - what type of deals would be possible for the service they are offering for example in real estate it could be multifamily / REIT / Syndicate, think according to the data point what could be the ideal type of deal] real fast.

		Would you like to know more?
	""",
  "WEBINAR": """
   	[FIRST NAME] - Just came across your [PERSONALISATION - Name/topic of the webinar] webinar — the focus on [PERSONALISATION - Some insight which would fit in perfectly with this sentence and tone] sounds like a game-changer!

		This kind of value deserves a much bigger audience.
        
		Assuming we could 10X your exposure in 45 days, without you lifting a finger…

    We’ll do this by using the insights from your [PERSONALISATION - Name/topic of the webinar] webinar and featuring them on our network of High Traffic Pages.
		
		It can build authority and turn strangers into buyers fast, for your paid programs.
		
		One client saw a 287% Increase in online course purchases.
		
		Would you like to know how?
	""",
  "PODCAST": """
    [FIRST NAME] - Just came across your podcast about [PERSONALISATION - some insight from the podcast which really stood out and fits here in the sentence] – honestly, so good!

		The pod deserves a much bigger audience for so much value. 

		What if we could 10X your exposure in 45 days, no effort on your part?

		We’d pull insights from the pod & feature them on our high-traffic network.

		This builds authority and turns strangers into buyers for your paid programs fast. One client saw a 287% Spike in course sales.

		Want to hear how it works?
	""",
	"EBOOK": """
    [FIRST NAME] - Saw your Book on [PERSONALISATION - What is the ebook about, deduce from the title or the description] — I think it’s a great way to get people into your higher ticket stuff.

		Would be great if more people saw it.

		If we could use it to 10X your exposure in 45 days, without you lifting a finger…

		We’ll do this by using the insights from your book and feature them on our network of High Traffic Pages.

		It can build authority and turn strangers into buyers fast.

		One client saw a 287% Increase in online course purchases. 

		Would you like to know how?
	""",
	"RECENT_BLOG": """
    [FIRST NAME] - Saw your blog on [PERSONALISATION - What is the blog about] — really hits the mark.

		Would be great if more people read it.

		If we could use it to 10X your exposure in 45 days, without you lifting a finger…

		We’ll do this by using the insights from your blog and feature them on our network of High Traffic Pages.

		It can build authority and turn strangers into buyers fast, for your higher ticket programs.

		One client saw a 287% Increase in online course purchases. 

		Would you like to know how?
	""",
	"ABOUT_US": """
    [FIRST NAME] - I looked into [PERSONALISATION - Company Name] and love your [PERSONALISATION - What they do and why we like that - insightful].

		Feels like, a ton of people could use that.

		Assuming, we could take your existing stuff & 5X your existing exposure in 45 days…

		Without you doing any heavy lifting.


		I think there’s so much value here, the right people would straight-up flock to it.

		One client saw a 287% Increase in online course purchases. 

		Would you like to know more?
	""",
	"NEUTRAL": """
    [FIRST NAME] – being someone with your own course/consulting offer.

		I can see how more visibility can help sell more of those.

		Assuming, we could take your existing stuff & 5X your existing exposure in 45 days…

		Without you doing any heavy lifting.

		So much value here, the right people would straight-up flock to it.

		One client saw a 287% Increase in online course purchases / consulting clients. 

		Would you like to know more?
	"""
},
"Subsequence 1": {
	"TESTIMONIALS": """
    Thanks for reaching out! The work you did with [PERSONALISATION - Jason and how you helped him 10x his revenue] actually made me think…

		Can you help more folks make the same leap? Pretty sure more people would want that!

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"COURSES": """
    Thanks for reaching out! Your [PERSONALISATION - Name of the course or the program, suffix with the word course or program as per the data point and if it feels necessary] actually made me think…

		Can we use some insights from it too? Folks would be all over it!

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"SERVICES": """
    Thanks for reaching out! Your [PERSONALISATION - some insight that made us think should fit in with the rest of the sentence here] page actually made me think…

		Mind if we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"WEBINAR": """
		Thanks for reaching out! Your [PERSONALISATION - Name/topic of the webinar, remove the word webinar if it is there at the end of the name] webinar actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"PODCAST": """
    Thanks for reaching out! Your podcast about [PERSONALISATION - some insight from the podcast which really stood out and fits here in the sentence] actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"EBOOK": """
    Thanks for reaching out! Your Book on [PERSONALISATION - What is the ebook about] actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"RECENT_BLOG": """
    Thanks for reaching out! Your blog on [PERSONALISATION - What is the blog about] actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"ABOUT_US": """
    Thanks for reaching out! I looked into [COMPANY NAME] and loved your approach [PERSONALISATION - What they do and why we like that - insightful].

		Feels like a ton of people could use that.

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"NEUTRAL": """
    Appreciate you reaching back out! I think there’s sooo much potential if we do this together.

		Here’s a quick 2 min pre-recorded video (to save myself some time LOL) we recorded going into this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 5-10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	"""
},
"Subsequence 2": """
	I was looking at your [Website] and couldn’t help but notice [PERSONALISATION - Some insight related to how they are helping their ICP]. 
    
	Got some time to talk about it tomorrow?
""",
"Subsequence 3": """
	Knowing that AI’s blowing up in [Industry], [First Name].
    
	Like those crazy chatbots!
    
	I'm curious, what’s your next move to stay ahead of the curve?
""",
"Subsequence 4": """
	I keep wondering about how [Company Name] is pushing [PERSONALISATION - broad topic, e.g., customer engagement].
    
	We’ve been working with some folks on similar goals, and I’d love to bounce a couple ideas your way—might be a fit. 
    
	Got a minute to talk this week?
""",
"Email 2": """
	Hate to bug you. Is this something I can pass along or no?
""",
"Email 3": """
	Hey [First Name],

	Just wanted to let you know…

	Being an Invite only firm, part of our offer is:

	If we can’t 5x your current exposure in the next 45 days, then we work for FREE until we do.

	Would it make sense to talk about it?
"""
}

BASE_PROMPT = """
    Objective: Generate initial cold emails for outreach, following the specific template provided below. Only modify the sections within square brackets for personalisation; all other content should remain fixed.

    Instructions:

    1. Personalization Fields:
    - Replace [FIRST NAME] with the name of the person given in the prompt.
    - Replace [PERSONALISATION] with a short, specific comment as per the instruction given in square bracket of [PERSONALISATION - instruction here].
    - While personalizing: Write the personalization in 3rd Grade level. The sentence should not be too long and complex. Use shorter sentences and simpler words.
 
    2. Fixed Content:
    - Do not change any other text in the template. All non-bracketed content should remain exactly as written, preserving the wording, tone, and format. Be very very strict on this, I don't want anything else apart from the bracketed  content to change. 

    3. Tone and Language:
    - Keep the tone friendly and professional.
    - Ensure the language is simple, conversational, and concise to stay within a ~150-word limit.

    4. Dont send anything else except for the Email
"""

In [8]:

def authenticate_google_sheets():
    """Authenticate and return Google Sheets service"""
    try:
        creds = service_account.Credentials.from_service_account_file(
            CREDENTIALS_FILE, 
            scopes=['https://www.googleapis.com/auth/spreadsheets']
        )
        service = build('sheets', 'v4', credentials=creds)
        return service
    except Exception as e:
        print(f"Error authenticating Google Sheets: {e}")
        return None

def initialize_openai():
    """Initialize OpenAI client"""
    try:
        client = OpenAI(api_key=OPENAI_API_KEY)
        return client
    except Exception as e:
        print(f"Error initializing OpenAI: {e}")
        return None

def count_words(text):
    """Count words in a text string"""
    if not text or text.strip() == "":
        return 0
    return len(text.strip().split())

def collect_datapoints(row):
    """Collect all datapoints from a row with their rankings"""
    datapoints = []
    
    for category, col_index in category_indices.items():
        if category == "SHOP":  # Exclude SHOP from processing
            continue
            
        if col_index < len(row):
            content = str(row[col_index]).strip()
            word_count = count_words(content)
            
            # Only include datapoints with more than 10 words
            if word_count > 10:
                datapoints.append({
                    'category': category,
                    'content': content,
                    'rank': category_ranking[category],
                    'word_count': word_count
                })
    
    # Sort by rank (ascending order - lower rank = higher priority)
    datapoints.sort(key=lambda x: x['rank'])
    return datapoints

def assign_datapoints_to_emails(datapoints):
    """Assign datapoints to emails based on ranking"""
    email_assignments = {
        'Email 1': None,
        'Subsequence 1': None,
        'Subsequence 2': None,
        'Subsequence 3': None,
        'Subsequence 4': None
    }
    
    email_order = ['Email 1', 'Subsequence 1', 'Subsequence 2', 'Subsequence 3', 'Subsequence 4']
    
    # Assign datapoints in order of their ranking
    for i, email_type in enumerate(email_order):
        if i < len(datapoints):
            email_assignments[email_type] = datapoints[i]
    
    return email_assignments

def generate_email_with_datapoint(email_type, category, datapoint_content, first_name, company_name, client):
    """Generate email using OpenAI for emails with datapoints"""
    try:
        # Handle different template structures
        if email_type in ["Email 1", "Subsequence 1"]:
            template = prompt_templates[email_type][category]
        else:
            # For Subsequence 2, 3, 4 - they have simple string templates
            template = prompt_templates[email_type]
        
        prompt = f"""
        {BASE_PROMPT}
        
        Template to use:
        {template}
        
        Person's first name: {first_name}
        Company name: {company_name}
        Datapoint content to personalize with: {datapoint_content}
        Category: {category}
        
        Generate the email following the template exactly, only replacing the bracketed placeholders.
        """
        
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
            temperature=0.7
        )
        
        return response.choices[0].message.content.strip()
    
    except Exception as e:
        print(f"Error generating email with datapoint: {e}")
        return f"Error generating {email_type}"

def generate_neutral_email(email_type, first_name, company_name, website="", industry="", client=None):
    """Generate neutral email for emails without datapoints"""
    try:
        if email_type in ["Email 1", "Subsequence 1"]:
            template = prompt_templates[email_type]["NEUTRAL"]
            
            if client:
                prompt = f"""
                {BASE_PROMPT}
                
                Template to use:
                {template}
                
                Person's first name: {first_name}
                Company name: {company_name}
                
                Generate the email following the template exactly, only replacing the bracketed placeholders.
                """
                
                response = client.chat.completions.create(
                    model="gpt-4",
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=500,
                    temperature=0.7
                )
                
                return response.choices[0].message.content.strip()
            else:
                # Simple replacement for neutral templates
                email = template.replace("[FIRST NAME]", first_name)
                email = email.replace("[COMPANY NAME]", company_name)
                return email.strip()
        
        elif email_type in ["Subsequence 2", "Subsequence 3", "Subsequence 4"]:
            # No neutral templates provided for these subsequences
            return "NO template provided"
        
        return f"Neutral template for {email_type}"
    
    except Exception as e:
        print(f"Error generating neutral email: {e}")
        return f"Error generating neutral {email_type}"

def generate_simple_email(email_type, first_name, company_name=""):
    """Generate simple emails for Email 2 and Email 3"""
    template = prompt_templates.get(email_type, "")
    email = template.replace("[First Name]", first_name)
    email = email.replace("[COMPANY NAME]", company_name)
    return email.strip()

def process_row(row, row_index, client):
    """Process a single row and generate all emails"""
    try:
        # Extract basic information (assuming standard positions)
        first_name = str(row[1]).strip() if len(row) > 1 else "Friend"  # Column B
        company_name = str(row[2]).strip() if len(row) > 2 else "Your Company"  # Column C
        website = str(row[3]).strip() if len(row) > 3 else ""  # Column D
        industry = str(row[4]).strip() if len(row) > 4 else ""  # Column E
        
        print(f"Processing row {row_index + 1}: {first_name} from {company_name}")
        
        # Step 1: Collect all datapoints
        datapoints = collect_datapoints(row)
        print(f"  Found {len(datapoints)} valid datapoints")
        
        # Step 2: Assign datapoints to emails
        email_assignments = assign_datapoints_to_emails(datapoints)
        
        # Step 3: Generate emails
        results = {}
        
        # Generate Email 1 and Subsequence 1
        for email_type in ['Email 1', 'Subsequence 1']:
            assignment = email_assignments[email_type]
            if assignment:
                # Generate email with datapoint
                email_content = generate_email_with_datapoint(
                    email_type, 
                    assignment['category'], 
                    assignment['content'],
                    first_name,
                    company_name,
                    client
                )
                results[email_type] = email_content
                results[f"{email_type} Data Point"] = f"{assignment['category']}: {assignment['content'][:100]}..."
            else:
                # Generate neutral email
                email_content = generate_neutral_email(
                    email_type, 
                    first_name, 
                    company_name,
                    website,
                    industry,
                    client
                )
                results[email_type] = email_content
                results[f"{email_type} Data Point"] = "No data point"
        
        # Generate Subsequence 2, 3, 4
        for email_type in ['Subsequence 2', 'Subsequence 3', 'Subsequence 4']:
            assignment = email_assignments[email_type]
            if assignment:
                # Generate email with datapoint using AI
                email_content = generate_email_with_datapoint(
                    email_type,
                    assignment['category'],
                    assignment['content'],
                    first_name,
                    company_name,
                    client
                )
                results[email_type] = email_content
                results[f"{email_type} Data Point"] = f"{assignment['category']}: {assignment['content'][:100]}..."
            else:
                # No neutral templates provided for these subsequences
                results[email_type] = "NO template provided"
                results[f"{email_type} Data Point"] = "No data point"
        
        # Generate Email 2 and Email 3
        results['Email 2'] = generate_simple_email('Email 2', first_name, company_name)
        results['Email 3'] = generate_simple_email('Email 3', first_name, company_name)
        
        return results
    
    except Exception as e:
        print(f"Error processing row {row_index + 1}: {e}")
        return None

def column_letter_to_number(letter):
    """Convert column letter to number (A=1, B=2, etc.)"""
    result = 0
    for char in letter:
        result = result * 26 + (ord(char.upper()) - ord('A') + 1)
    return result

def update_sheet_with_results(service, results_list):
    """Update the Google Sheet with generated emails"""
    try:
        # Prepare batch update data
        data = []
        
        for row_index, results in enumerate(results_list):
            if results is None:
                continue
                
            actual_row = row_index + 2  # Assuming header is row 1, data starts from row 2
            
            for email_type, column_letter in output_columns.items():
                if email_type in results:
                    cell_range = f"{SHEET_NAME}!{column_letter}{actual_row}"
                    data.append({
                        'range': cell_range,
                        'values': [[results[email_type]]]
                    })
        
        if data:
            body = {
                'valueInputOption': 'RAW',
                'data': data
            }
            
            result = service.spreadsheets().values().batchUpdate(
                spreadsheetId=SPREADSHEET_ID,
                body=body
            ).execute()
            
            print(f"Updated {len(data)} cells in the spreadsheet")
            return True
        else:
            print("No data to update")
            return False
    
    except Exception as e:
        print(f"Error updating sheet: {e}")
        return False

def main():
    """Main function to run the email generation process"""
    print("Starting email generation process...")
    
    # Initialize services
    sheets_service = authenticate_google_sheets()
    if not sheets_service:
        print("Failed to authenticate Google Sheets")
        return
    
    openai_client = initialize_openai()
    if not openai_client:
        print("Failed to initialize OpenAI client")
        return
    
    try:
        # Read data from Google Sheets
        range_name = f"{SHEET_NAME}!A:U"  # Read all data up to column U
        result = sheets_service.spreadsheets().values().get(
            spreadsheetId=SPREADSHEET_ID,
            range=range_name
        ).execute()
        
        values = result.get('values', [])
        
        if not values:
            print('No data found in the sheet.')
            return
        
        print(f"Found {len(values)} rows in the sheet")
        
        # Process each row (skip header row)
        results_list = []
        for i, row in enumerate(values[1:]):  # Skip header row
            results = process_row(row, i, openai_client)
            results_list.append(results)
            
            # Optional: Add delay to avoid rate limiting
            import time
            time.sleep(1)
        
        # Update the sheet with results
        if update_sheet_with_results(sheets_service, results_list):
            print("Email generation completed successfully!")
        else:
            print("Email generation completed but failed to update sheet")
    
    except Exception as e:
        print(f"Error in main process: {e}")

if __name__ == "__main__":
    main()

Starting email generation process...
Found 10 rows in the sheet
Processing row 1: Ludeking from President
  Found 0 valid datapoints
Processing row 2: Dibartolomeo from CEO
  Found 3 valid datapoints
Processing row 3: Schibell from Owner and Tax Partner, CPA,CFP, PFS
  Found 7 valid datapoints
Processing row 4: Woods from Chief Executive Officer
  Found 6 valid datapoints
Processing row 5: Singh from CEO & Co-Founder
  Found 2 valid datapoints
Processing row 6: Thota from Founder and Principal Consultant
  Found 6 valid datapoints
Processing row 7: Lynch from Founder
  Found 3 valid datapoints
Processing row 8: Franco from President And Chief Executive Officer
  Found 2 valid datapoints
Processing row 9: Coppinger from Owner
  Found 6 valid datapoints
Updated 108 cells in the spreadsheet
Email generation completed successfully!
